# Day 01 — Endüstriyel Yapay Zeka Geliştirme Ortamı ve Tekrarlanabilirlik

Bu notebook, 40 günlük **Endüstriyel Yapay Zeka Staj Portföyü**nün ilk gününde geliştirme ortamının deterministik, tekrarlanabilir ve izole biçimde nasıl kurgulanacağını inceler.

---


## 1. Problem

Yapay zeka ve derin öğrenme projelerinde en sık karşılaşılan engellerin başında **çalışma ortamı tutarsızlıkları** (dependency drift) ve **donanım uyumsuzlukları** gelir. Bir geliştiricinin yerel makinesinde hatasız çalışan bir model pipeline'ı; sunucuya, Docker konteynerine veya üretim hattındaki bir endüstriyel uç cihaza (edge device) aktarıldığında Python alt sürümleri, CUDA sürücüleri veya C++ kütüphane derleme farkları nedeniyle çökebilir.


## 2. Why the Problem Matters (Bu Problem Neden Önemli?)

Endüstriyel üretim işletmelerinde (ör. tekstil ve halı fabrikalarında) modeller yalnız araştırma amaçlı değil; fabrika ERP, MES ve otomatik dokuma tezgahlarıyla entegre çalışan canlı servisler olarak konumlanır.

- **Zaman Kaybı:** 'Benim bilgisayarımda çalışıyordu' argümanı haftalarca süren dağıtım gecikmelerine yol açar.
- **Determinizm Kaybı:** Rastgelelik çekirdeklerinin (seeds) ve kütüphane sürümlerinin kilitlenmemesi, model ağırlıklarının ve test sonuçlarının tekrarlanamaz hale gelmesine neden olur.
- **Donanım Darboğazı:** CUDA sürüm uyumsuzluğu durumunda GPU hızlandırmasının sessizce CPU fallback'e düşmesi inference sürelerini 50 kat uzatabilir.


## 3. Engineering Concepts (Mühendislik Kavramları)

1. **Sanal Ortam İzolasyonu (Virtual Environments):** Küresel sistem Python paketlerini kirletmeden projeye özel izole bağımlılık havuzu yönetimi.
2. **Deterministik Bağımlılık Kitleme (Lockfiles):** `poetry.lock` ile tüm ikincil ve üçüncül paket sürümlerinin ve sağlama toplamlarının (hashes) sabitlenmesi.
3. **Statik Kod Analizi ve Biçimlendirme:** Ruff ve Black ile kod tabanının tek tip stil ve tip güvenliğine kavuşturulması.
4. **Pre-commit Kancaları (Hooks):** Hatalı veya biçimlendirilmemiş kodun Git geçmişine girmesinin yerel commit aşamasında engellenmesi.
5. **Donanım Soyutlama:** Modelin çalışacağı fiziksel GPU/CUDA katmanının çalışma zamanında otomatik olarak algılanıp raporlanması.


## 4. Library & API Investigation

Aşağıdaki hücrede ortam denetiminde kullandığımız temel standart kütüphaneleri (`sys`, `platform`, `importlib.metadata`) ve hızlandırıcı katmanı (`torch.cuda`) inceliyoruz:


In [1]:
import sys
import platform
import importlib.metadata

print(f"Python Sürümü   : {sys.version}")
print(f"İşletim Sistemi : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Çalıştırıcı     : {sys.executable}")

try:
    import torch
    print(f"PyTorch Sürümü  : {torch.__version__}")
    print(f"CUDA Aktif mi?  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Cihazı      : {torch.cuda.get_device_name(0)}")
        print(f"CUDA Sürümü     : {torch.version.cuda}")
except ImportError:
    print("PyTorch henüz kurulu değil.")


Python Sürümü   : 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
İşletim Sistemi : Windows 11 (AMD64)
Çalıştırıcı     : C:\Users\Seydi Eryılmaz\AppData\Local\Programs\Python\Python314\python.exe


PyTorch Sürümü  : 2.11.0+cu126
CUDA Aktif mi?  : True
GPU Cihazı      : NVIDIA GeForce RTX 4070 Laptop GPU
CUDA Sürümü     : 12.6


## 5. Minimal Implementation

Üretim kodunu notebook hücrelerinde bırakmak yerine `mini_project/src/env_checker.py` modülü altında yeniden kullanılabilir sınıflara (`EnvironmentChecker`, `EnvironmentReport`) dönüştürdük. Şimdi bu modülü çağırıyoruz:


In [2]:
from pathlib import Path
import sys

# Kök ve modül dizini dinamik çözümü
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.startswith('day') else CURRENT_DIR
DAY01_DIR = PROJECT_ROOT / 'day01'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

module_path = (DAY01_DIR / 'mini_project' / 'src').resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from env_checker import EnvironmentChecker

spec_path = DAY01_DIR / 'mini_project' / 'configs' / 'env_spec.json'
checker = EnvironmentChecker(spec_path=spec_path)
print('EnvironmentChecker başarıyla yüklendi.')


EnvironmentChecker başarıyla yüklendi.


## 6. Experiment

Yerel geliştirme ortamının tam taramasını gerçekleştiriyor ve yapısal ortam raporunu alıyoruz:


In [3]:
report = checker.run_full_check()
print(f"Sistem Uyumluluk Durumu: {'UYUMLU' if report.is_compliant else 'UYUMSUZ'}")
for msg in report.validation_messages:
    print(f"  - {msg}")


2026-09-23 00:55:03,591 [INFO] EnvironmentChecker: Ortam denetimi başlatılıyor...


2026-09-23 00:55:03,599 [INFO] EnvironmentChecker: Ortam denetimi tamamlandı. Uyumluluk durumu: True


Sistem Uyumluluk Durumu: UYUMLU
  - Python 3.14 tespit edildi (Beklenen asgari: 3.11).
  - Kurulu zorunlu paket: numpy (2.4.3)
  - Kurulu zorunlu paket: pydantic (2.13.3)
  - Kurulu zorunlu paket: pytest (9.0.3)
  - Kurulu opsiyonel paket: torch (2.11.0+cu126)
  - Kurulu opsiyonel paket: torchvision (0.26.0+cu126)
  - Kurulu opsiyonel paket: fastapi (0.139.0)
  - Kurulu opsiyonel paket: redis (6.4.0)
  - Kurulu opsiyonel paket: celery (5.6.3)


## 7. Visualization

Denetlenen paketlerin kurulum durumlarını ve donanım özetini Pandas tablosu olarak görselleştiriyoruz:


In [4]:
import pandas as pd

pkg_data = [
    {
        "Paket Adı": p.name,
        "Kurulu mu?": "Evet" if p.installed else "Hayır",
        "Tespit Edilen Sürüm": p.version if p.version else "-",
        "Gereksinim": "Zorunlu" if p.required else "Opsiyonel"
    }
    for p in report.packages
]
df_packages = pd.DataFrame(pkg_data)
display(df_packages)


,Paket Adı,Kurulu mu?,Tespit Edilen Sürüm,Gereksinim
0,numpy,Evet,2.4.3,Zorunlu
1,pydantic,Evet,2.13.3,Zorunlu
2,pytest,Evet,9.0.3,Zorunlu
3,torch,Evet,2.11.0+cu126,Opsiyonel
4,torchvision,Evet,0.26.0+cu126,Opsiyonel
5,cv2,Hayır,-,Opsiyonel
6,fastapi,Evet,0.139.0,Opsiyonel
7,redis,Evet,6.4.0,Opsiyonel
8,celery,Evet,5.6.3,Opsiyonel


## 8. Validation

Asgari Python ve paket kriterlerinin karşılandığını iddia (assertion) mekanizmalarıyla teyit ediyoruz:


In [5]:
assert sys.version_info[:2] >= (3, 11), "Python sürümü 3.11 veya üzeri olmalıdır!"
assert any(p.name == "numpy" and p.installed for p in report.packages), "NumPy kurulu olmalıdır!"
assert any(p.name == "pytest" and p.installed for p in report.packages), "Pytest kurulu olmalıdır!"
print("Tüm doğrulama kuralları başarıyla geçti!")


Tüm doğrulama kuralları başarıyla geçti!


## 9. Failure Cases (Hata Senaryoları)

Eksik veya var olmayan bir paket istendiğinde veya konfigürasyon dosyası bozulduğunda sistemin davranışı:


In [6]:
# Olmayan bir paket denetlendiğinde sistem çökmemeli, PackageStatus.installed=False dönmeli
missing_pkg = checker.check_package("non_existent_fake_package")
print(f"Olmayan paket durumu: {missing_pkg}")
assert missing_pkg.installed is False
assert missing_pkg.version is None
print("Hata durumu beklendiği gibi zarifçe (gracefully) ele alındı.")


Olmayan paket durumu: PackageStatus(name='non_existent_fake_package', installed=False, version=None, required=True)
Hata durumu beklendiği gibi zarifçe (gracefully) ele alındı.


## 10. Conclusions (Sonuç ve Day 02 Hazırlığı)

Day 01 kapsamında:
1. Tekrarlanabilir ve kurumsal bir repository yapısı bootstrap edildi.
2. Python 3.14 ve CUDA 12.6 destekli NVIDIA GeForce RTX 4070 Laptop GPU donanım kaynakları doğrulandı.
3. Mini proje (`environment-bootstrap`) birim testlerle desteklendi.

**Sonraki Gün (Day 02):** Problem uzayının tanımlanması; endüstriyel görsel, ürün metadata ve inference veri modellerinin Pydantic v2 ile kurgulanması ele alınacaktır.
